In [1]:
import os, sys
import numpy             as np
import xarray            as xr
import xesmf             as xe
import pandas            as pd
import dask.array        as da
import dask.dataframe    as dd
import matplotlib.pyplot as plt
import cmocean           as cm
import cartopy.crs       as ccrs
import cartopy.feature   as cft
import matplotlib.path   as mpath
from dask.diagnostics    import ProgressBar
from dask.distributed    import Client
from datetime            import datetime

In [ ]:
def calculate_corners(center_lat, center_lon):
    """Calculate corner coordinates by averaging neighbor cells
    """

    # get rank
    rank = len(center_lat.dims)

    if rank == 1:
        # get dimensions
        nlon = center_lon.size
        nlat = center_lat.size

        # convert center points from 1d to 2d
        center_lat2d = da.broadcast_to(center_lat.values[None,:], (nlon, nlat))
        center_lon2d = da.broadcast_to(center_lon.values[:,None], (nlon, nlat))
    elif rank == 2:
        # get dimensions
        dims = center_lon.shape
        nlon = dims[0]
        nlat = dims[1] 

        # just rename and convert to dask array
        center_lat2d = da.from_array(center_lat)
        center_lon2d = da.from_array(center_lon)
    else:
        print('Unrecognized grid! The rank of coordinate variables can be 1 or 2 but it is {}.'.format(rank))
        sys.exit(2)

    # calculate corner coordinates for latitude, counterclockwise order, imposing Fortran ordering
    center_lat2d_ext = da.from_array(np.pad(center_lat2d.compute(), (1,1),  mode='reflect', reflect_type='odd'))

    ur = (center_lat2d_ext[1:-1,1:-1]+
          center_lat2d_ext[0:-2,1:-1]+
          center_lat2d_ext[1:-1,2:]+
          center_lat2d_ext[0:-2,2:])/4.0
    ul = (center_lat2d_ext[1:-1,1:-1]+
          center_lat2d_ext[0:-2,1:-1]+
          center_lat2d_ext[1:-1,0:-2]+
          center_lat2d_ext[0:-2,0:-2])/4.0
    ll = (center_lat2d_ext[1:-1,1:-1]+
          center_lat2d_ext[1:-1,0:-2]+
          center_lat2d_ext[2:,1:-1]+
          center_lat2d_ext[2:,0:-2])/4.0
    lr = (center_lat2d_ext[1:-1,1:-1]+
          center_lat2d_ext[1:-1,2:]+
          center_lat2d_ext[2:,1:-1]+
          center_lat2d_ext[2:,2:])/4.0

    # this looks clockwise ordering but it is transposed and becomes counterclockwise, bit-to-bit with NCL 
    corner_lat = da.stack([ul.T.reshape((-1,)).T, ll.T.reshape((-1,)).T, lr.T.reshape((-1,)).T, ur.T.reshape((-1,)).T], axis=1)

    # calculate corner coordinates for longitude, counterclockwise order, imposing Fortran ordering
    center_lon2d_ext = da.from_array(np.pad(center_lon2d.compute(), (1,1),  mode='reflect', reflect_type='odd'))

    ur = (center_lon2d_ext[1:-1,1:-1]+
          center_lon2d_ext[0:-2,1:-1]+
          center_lon2d_ext[1:-1,2:]+
          center_lon2d_ext[0:-2,2:])/4.0
    ul = (center_lon2d_ext[1:-1,1:-1]+
          center_lon2d_ext[0:-2,1:-1]+
          center_lon2d_ext[1:-1,0:-2]+
          center_lon2d_ext[0:-2,0:-2])/4.0
    ll = (center_lon2d_ext[1:-1,1:-1]+
          center_lon2d_ext[1:-1,0:-2]+
          center_lon2d_ext[2:,1:-1]+
          center_lon2d_ext[2:,0:-2])/4.0
    lr = (center_lon2d_ext[1:-1,1:-1]+
          center_lon2d_ext[1:-1,2:]+
          center_lon2d_ext[2:,1:-1]+
          center_lon2d_ext[2:,2:])/4.0

    # this looks clockwise ordering but it is transposed and becomes counterclockwise, bit-to-bit with NCL 
    corner_lon = da.stack([ul.T.reshape((-1,)).T, ll.T.reshape((-1,)).T, lr.T.reshape((-1,)).T, ur.T.reshape((-1,)).T], axis=1)

    return center_lat2d, center_lon2d, corner_lat, corner_lon 

In [ ]:
def write_to_esmf_mesh(filename, center_lat, center_lon, corner_lat, corner_lon, mask, area=None):
    """
    Writes ESMF Mesh to file
    dask array doesn't support order='F' for Fortran-contiguous (row-major) order
    the workaround is to arr.T.reshape.T
    """
    # create array with unique coordinate pairs
    # remove coordinates that are shared between the elements
    corner_pair = da.stack([corner_lon.T.reshape((-1,)).T, corner_lat.T.reshape((-1,)).T], axis=1)

    # REPLACED: corner_pair_uniq = dd.from_dask_array(corner_pair).drop_duplicates().to_dask_array(lengths=True)
    # following reduces memory by %17
    corner_pair_uniq = dd.from_dask_array(corner_pair).drop_duplicates().values
    corner_pair_uniq.compute_chunk_sizes()

    # check size of unique coordinate pairs
    dims = mask.shape
    nlon = dims[0]
    nlat = dims[1]
    elem_conn_size = nlon*nlat+nlon+nlat+1
    if corner_pair_uniq.shape[0] != elem_conn_size:
        print('The size of unique coordinate pairs is {} but expected size is {}!'.format(corner_pair_uniq.shape[0], elem_conn_size))
        print('Please check the input file or try to force double precision with --double option. Exiting ...')
        sys.exit(2)

    # create element connections
    corners = dd.concat([dd.from_dask_array(c) for c in [corner_lon.T.reshape((-1,)).T, corner_lat.T.reshape((-1,)).T]], axis=1)   
    corners.columns = ['lon', 'lat']
    elem_conn = corners.compute().groupby(['lon','lat'], sort=False).ngroup()+1
    elem_conn = da.from_array(elem_conn.to_numpy())

    # create new dataset for output
    out = xr.Dataset()

    out['origGridDims'] = xr.DataArray(np.array(center_lon.shape, dtype=np.int32),
                                       dims=('origGridRank'))

    out['nodeCoords'] = xr.DataArray(corner_pair_uniq,
                                     dims=('nodeCount', 'coordDim'),
                                     attrs={'units': 'degrees'})

    out['elementConn'] = xr.DataArray(elem_conn.T.reshape((4,-1)).T,
                                      dims=('elementCount', 'maxNodePElement'),
     		                      attrs={'long_name': 'Node indices that define the element connectivity'})
    out.elementConn.encoding = {'dtype': np.int32}

    out['numElementConn'] = xr.DataArray(4*np.ones(center_lon.size, dtype=np.int32),
                                         dims=('elementCount'),
                                         attrs={'long_name': 'Number of nodes per element'})

    out['centerCoords'] = xr.DataArray(da.stack([center_lon.T.reshape((-1,)).T,
                                                 center_lat.T.reshape((-1,)).T], axis=1),
                                          dims=('elementCount', 'coordDim'),
                                          attrs={'units': 'degrees'})

    # add area if it is available
    if area:
        out['elementArea'] = xr.DataArray(area.T.reshape((-1,)).T,
                                          dims=('elementCount'),
                                          attrs={'units': 'radians^2',
                                                 'long_name': 'area weights'})     

    # add mask
    out['elementMask'] = xr.DataArray(mask.T.reshape((-1,)).T,
                                      dims=('elementCount'),
                                      attrs={'units': 'unitless'})
    out.elementMask.encoding = {'dtype': np.int32}

    # force no '_FillValue' if not specified
    for v in out.variables:
        if '_FillValue' not in out[v].encoding:
            out[v].encoding['_FillValue'] = None

    # add global attributes
    out.attrs = {'title': 'ESMF unstructured grid file for rectangular grid with {} dimension'.format('x'.join(list(map(str,center_lat.shape)))),
                 'created_by': "dpath2o, daniel.atwater@utas.edu.au",
                 'date_created': '{}'.format(datetime.now()),
                 'conventions': 'ESMFMESH',
                }

    # write output file
    if filename is not None:
        print('Writing {} ...'.format(filename))
        out.to_netcdf(filename)

In [ ]:
def write_to_scrip(filename, center_lat, center_lon, corner_lat, corner_lon, mask, area=None):
    """
    Writes SCRIP grid definition to file
    dask array doesn't support order='F' for Fortran-contiguous (row-major) order
    the workaround is to arr.T.reshape.T
    """
    # create new dataset for output 
    out = xr.Dataset()

    out['grid_dims'] = xr.DataArray(np.array(center_lat.shape, dtype=np.int32), 
                                    dims=('grid_rank',)) 
    out.grid_dims.encoding = {'dtype': np.int32}

    out['grid_center_lat'] = xr.DataArray(center_lat.T.reshape((-1,)).T, 
                                          dims=('grid_size'),
                                          attrs={'units': 'degrees'})

    out['grid_center_lon'] = xr.DataArray(center_lon.T.reshape((-1,)).T, 
                                          dims=('grid_size'),
                                          attrs={'units': 'degrees'})

    out['grid_corner_lat'] = xr.DataArray(corner_lat.T.reshape((4, -1)).T,
                                          dims=('grid_size','grid_corners'),
                                          attrs={'units': 'degrees'})

    out['grid_corner_lon'] = xr.DataArray(corner_lon.T.reshape((4, -1)).T,
                                          dims=('grid_size','grid_corners'),
                                          attrs={'units': 'degrees'})

    # include area if it is available
    if area:
        out['grid_area'] = xr.DataArray(area.T.reshape((-1,)).T, 
                                        dims=('grid_size'),
                                        attrs={'units': 'radians^2',
                                               'long_name': 'area weights'})

    out['grid_imask'] = xr.DataArray(mask.T.reshape((-1,)).T, 
                                     dims=('grid_size'),
                                     attrs={'units': 'unitless'})
    out.grid_imask.encoding = {'dtype': np.int32}

    # force no '_FillValue' if not specified
    for v in out.variables:
        if '_FillValue' not in out[v].encoding:
            out[v].encoding['_FillValue'] = None

    # add global attributes
    out.attrs = {'title': 'Rectangular grid with {} dimension'.format('x'.join(list(map(str,center_lat.shape)))),
                 'created_by': "dpath2o, daniel.atwater@utas.edu.au",
                 'date_created': '{}'.format(datetime.now()),
                 'conventions': 'SCRIP',
                }
    print(out)
    out = out.compute()
    # write output file
    if filename is not None:
        print('Writing {} ...'.format(filename))
        out.to_netcdf(filename) 

In [ ]:
def file_type(x):
    if x.lower() == 'scrip' or x.lower() == 'esmf':
       return x
    else:
        raise argparse.ArgumentTypeError('SCRIP or ESMF value expected for output type.')

In [ ]:
ifile = '/home/581/da1339/DATA/grids/ERA5_grid.nc'
ofile = '/home/581/da1339/DATA/grids/ERA5_grid.SCRIP.20231102.nc'
oformat = 'SCRIP'
overwrite = True
flip  = False
latrev = False
latvar = 'lat'
lonvar = 'lon'
maskvar = 'mask'
maskcal = False
addarea = False
double = False
print("Configuration:")
print("ifile     = {}".format(ifile))
print("ofile     = {}".format(ofile))
print("oformat   = {}".format(oformat))
print("overwrite = {}".format(overwrite))
print("flip      = {}".format(flip))
print("latrev    = {}".format(latrev))
print("latvar    = {}".format(latvar))
print("lonvar    = {}".format(lonvar))
print("maskvar   = {}".format(maskvar))
print("maskcal   = {} ({})".format(maskcal, maskvar))
print("addarea   = {}".format(addarea))
print("double    = {}".format(double))
# open file, transpose() fixes dimension ordering and mimic Fortran
if os.path.isfile(ifile):
    ds = xr.open_dataset(ifile, mask_and_scale=False, decode_times=False).transpose()
else:
    print('Input file could not find!') 
    sys.exit(2)       

# check output file
if overwrite:
    if os.path.isfile(ofile):
        print('Removing existing output file {}.'.format(ofile))
        os.remove(ofile)
else:
    if os.path.isfile(ofile):
        print('Output file exists. Please provide --overwrite flag.') 
        sys.exit(2)

# check coordinate variables
if latvar not in ds.coords and latvar not in ds.data_vars:
    print('Input file does not have variable named {}.'.format(latvar))
    print('File has following {}'.format(ds.coords))
    print('File has following {}'.format(ds.data_vars))
    sys.exit(2)

if lonvar not in ds.coords and lonvar not in ds.data_vars:
    print('Input file does not have variable named {}.'.format(latvar))
    print('File has following {}'.format(ds.coords))
    print('File has following {}'.format(ds.data_vars))
    sys.exit(2)

# remove time dimension from coordinate variables
hasTime = 'time' in ds[latvar].dims
if hasTime:
    lat = ds[latvar][:,:,0]
else:
    lat = ds[latvar]

hasTime = 'time' in ds[lonvar].dims
if hasTime:
    lon = ds[lonvar][:,:,0]
else:
    lon = ds[lonvar]

# reverse latitude dimension
if latrev:
    lat_name = [x for x in lat.coords.dims if 'lat' in x]
    if lat_name:
        lat = lat.reindex({lat_name[0]: list(reversed(lat[lat_name[0]]))})

# remove time dimension from mask variable and optionally flip mask values
# this will also create artifical mask variable with all ones, if it is required
if maskvar in ds.data_vars:
    print('Using mask values from the file.')

    # check mask has time dimension or not
    hasTime = 'time' in ds[maskvar].dims
    if hasTime:
        mask = ds[maskvar][:,:,0]
    else:
        mask = ds[maskvar][:]

    # use variable to construct mask information
    if maskcal:
        fill_value = None
        if '_FillValue' in mask.attrs:
            fill_value = mask._FillValue
        elif 'missing_value' in mask.attrs:
            fill_value = mask.missing_value

        if fill_value:
            mask = da.from_array(xr.where(mask == fill_value, 0, 1).astype(dtype=np.int8))
else:
    print('Using artifical generated mask values, that are ones in everywhere.')
    if len(lat.dims) == 1:
        mask = da.from_array(np.ones((next(iter(lon.sizes.values())), next(iter(lat.sizes.values()))), dtype=np.int8))
    else:
        mask = da.from_array(np.ones(tuple(lat.sizes.values()), dtype=np.int8))

# flip mask values
if flip:
    print('Flipping mask values to 0 for land and 1 for ocean')
    mask = xr.where(mask > 0, 0, 1)

# calculate corner coordinates, center coordinates are converted to 2d if it is 1d
if double:
    center_lat, center_lon, corner_lat, corner_lon = calculate_corners(lat.astype(np.float64, copy=False), lon.astype(np.float64, copy=False))
else:
    center_lat, center_lon, corner_lat, corner_lon = calculate_corners(lat, lon)

# TODO: add support to calculate area
if addarea:
    print('The area calculation is not supported! --addarea is reserved for future use.')

# create output file
if oformat.lower() == 'scrip':
    write_to_scrip(ofile, center_lat, center_lon, corner_lat, corner_lon, mask)  
else:
    write_to_esmf_mesh(ofile, center_lat, center_lon, corner_lat, corner_lon, mask)

In [ ]:
F_t2m           = "/g/data/rt52/era5/single-levels/reanalysis/2t/2005/2t_era5_oper_sfc_20050101-20050131.nc"
F_wgt           = "/g/data/jk72/da1339/grids/weights/map_ERA5_to_CICE5_0p25_patch_extrap_neareststod.nc"
F_G_CICE5       = "/g/data/jk72/da1339/grids/CICE5_0p25_grid_with_t_deg.nc"
F_G_CICE5_SCRIP = "/g/data/jk72/da1339/grids/CICE5_SCRIP_0p25.nc"
t2m             = xr.open_dataset(F_t2m)
t2m             = t2m.chunk({'time':1})
G_CICE5         = xr.open_dataset(F_G_CICE5)
G_CICE5_SCRIP   = xr.open_dataset(F_G_CICE5_SCRIP)
wgt             = xr.open_dataset(F_wgt)

In [ ]:
reG = xe.Regridder(t2m, G_CICE5, 'patch', weights=F_wgt)

In [ ]:
t2m_reG = reG(t2m['t2m'])

In [ ]:
t2m_reG = t2m_reG.chunk({'time':1})

In [ ]:
client = Client(n_workers=28, threads_per_worker=1, memory_limit='200GB')

In [ ]:
airtmp = xr.Dataset(data_vars = { 'airtmp' : (("time","ny","nx"),t2m_reG.values) },
                    coords    = { 'time' : (("time"),t2m.time.values),
                                  'lat'  : (('ny','nx'), G_CICE5.lat.values),
                                  'lon'  : (('ny','nx'), G_CICE5.lon.values) } )

In [ ]:
airtmp

In [ ]:
airtmp = airtmp.chunk({'time':1,'ny':108,'ny':144})

In [ ]:
delayed_obj = airtmp.to_netcdf("manipulated-example-data.nc", compute=False)
with ProgressBar():
    results = delayed_obj.compute()

In [ ]:
airtmp2 = xr.open_dataset("/scratch/jk72/da1339/afim_input/ERA5/0p25/airtmp_01.nc")
airtmp2 = airtmp2.chunk({'time':1})

In [ ]:
airtmp  = xr.open_dataset("/home/581/da1339/airtmp_01.nc")
airtmp  = airtmp.chunk({'time':1})

In [ ]:
for i, time_step in enumerate(t2m.time):
    fig, axs = plt.subplots(nrows=1, ncols=3, subplot_kw={'projection': ccrs.SouthPolarStereo()}, figsize=(15, 9), constrained_layout=True)
    theta = np.linspace(0, 2 * np.pi, 100)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * radius + center)
    for ax in axs:
        ax.set_extent([0, 360, -90, -50], ccrs.PlateCarree())
        ax.set_boundary(circle, transform=ax.transAxes)
        ax.add_feature(cft.LAND, color='darkgrey')
        ax.add_feature(cft.COASTLINE, linewidth=0.5)
        ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=1, color='gray', alpha=0.5, linestyle='--')
        ax.gridlines(xlocs=np.arange(-180, 181, 30), ylocs=np.arange(-90, -49, 10), color='black', alpha=0.5, linestyle='--')
    # Check for missing values in the datasets and create a mask
    mask_t2m     = np.isnan(t2m.isel(time=i).t2m)
    mask_airtmp2 = np.isnan(airtmp2.isel(time=i).airtmp)
    mask_airtmp  = np.isnan(airtmp.isel(time=i).airtmp)
    cs0 = axs[0].contourf(t2m.longitude, t2m.latitude, t2m.isel(time=i).t2m - 273.15, cmap=cm.cm.thermal, levels=np.linspace(-50, 20, 100), transform=ccrs.PlateCarree())
    cs1 = axs[1].contourf(airtmp2.lon, airtmp2.lat, airtmp2.isel(time=i).airtmp - 273.15, cmap=cm.cm.thermal, levels=np.linspace(-50, 20, 100), transform=ccrs.PlateCarree())
    cs2 = axs[2].contourf(airtmp.lon, airtmp.lat, airtmp.isel(time=i).airtmp - 273.15, cmap=cm.cm.thermal, levels=np.linspace(-50, 20, 100), transform=ccrs.PlateCarree())
    #pcm0 = axs[0].pcolormesh(t2m.longitude, t2m.latitude, t2m.isel(time=i).t2m - 273.15, cmap=cm.cm.thermal, vmin=-50, vmax=20, transform=ccrs.PlateCarree())
    #pcm1 = axs[1].pcolormesh(airtmp2.lon, airtmp2.lat, airtmp2.isel(time=i).airtmp - 273.15, cmap=cm.cm.thermal, vmin=-50, vmax=20, transform=ccrs.PlateCarree())
    #pcm2 = axs[2].pcolormesh(airtmp.lon, airtmp.lat, airtmp.isel(time=i).airtmp - 273.15, cmap=cm.cm.thermal, vmin=-50, vmax=20, transform=ccrs.PlateCarree())
    lon_2d0, lat_2d0 = np.meshgrid(t2m.longitude, t2m.latitude)
    lon_2d1, lat_2d1 = np.meshgrid(airtmp2.lon, airtmp2.lat)
    axs[0].plot(lon_2d0[mask_t2m], lat_2d0[mask_t2m], 'ko', markersize=3, transform=ccrs.PlateCarree())
    axs[1].plot(lon_2d1[mask_airtmp2], lat_2d1[mask_airtmp2], 'ko', markersize=3, transform=ccrs.PlateCarree())
    axs[2].plot(lon_2d1[mask_airtmp], lat_2d1[mask_airtmp], 'ko', markersize=3, transform=ccrs.PlateCarree())
    cbar = plt.colorbar(pcm1, ax=axs.ravel().tolist(), label='temperature (C)', orientation='horizontal', pad=0.1)
    plt.suptitle("t2m -- left (original), middle (nco), right (xesmf)")
    time_label = datetime.utcfromtimestamp(time_step.item()).strftime('%Y-%m-%d_%H:%M:%S')
    plt.savefig(f"/g/data/jk72/da1339/GRAPHICAL/ERA5/0p25/regrid_comparison/sh/figure_{time_label}.png", bbox_inches='tight')
    plt.close('all')



In [3]:
CICE5 = xr.open_dataset("/g/data/jk72/da1339/grids/cice5_025_grid.nc")
CICE5['tlon'] = CICE5.tlon * 180/np.pi
CICE5['tlat'] = CICE5.tlat * 180/np.pi
CICE5['ulon'] = CICE5.ulon * 180/np.pi
CICE5['ulat'] = CICE5.ulat * 180/np.pi
CICE5['tlon'].attrs['units'] = 'degrees_east'
CICE5['tlat'].attrs['units'] = 'degrees_north'
CICE5['ulon'].attrs['units'] = 'degrees_east'
CICE5['ulat'].attrs['units'] = 'degrees_north'
CICE5.to_netcdf("/g/data/jk72/da1339/grids/CICE5_0p25_grid_in_degrees.nc")

In [4]:
CICE5

<xarray.Dataset>
Dimensions:  (ny: 1080, nx: 1440)
Dimensions without coordinates: ny, nx
Data variables:
    ulat     (ny, nx) float64 -81.02 -81.02 -81.02 -81.02 ... 65.29 65.18 65.08
    ulon     (ny, nx) float64 -280.0 -279.8 -279.5 -279.2 ... 80.0 80.0 80.0
    tlat     (ny, nx) float64 -81.08 -81.08 -81.08 -81.08 ... 65.24 65.13 65.03
    tlon     (ny, nx) float64 -279.9 -279.6 -279.4 -279.1 ... 80.0 80.0 80.0
    htn      (ny, nx) float64 ...
    hte      (ny, nx) float64 ...
    angle    (ny, nx) float64 ...
    angleT   (ny, nx) float64 ...
    tarea    (ny, nx) float64 ...
    uarea    (ny, nx) float64 ...

In [5]:
CICE5.ulat.max()

<xarray.DataArray 'ulat' ()>
array(90.)